In [1]:
!pip install transformers datasets ipywidgets peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.0 MB/s eta 0:00:00


In [2]:
import os
import math
import numpy as np
import torch
from tqdm.auto import tqdm 
from datetime import timedelta
import time
import gc
from peft import prepare_model_for_kbit_training
from torch.nn.utils import prune
from transformers import BitsAndBytesConfig

# Set memory optimization environment variable
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    TrainerCallback
)
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Free up memory before starting
torch.cuda.empty_cache()
gc.collect()

# 1. Load your text dataset from the Kaggle input path
with open('/kaggle/input/paper-batch/cleaned_batch1.md', 'r', encoding='utf-8') as f:
    corpus = f.read()

# 2. Load the tokenizer and determine the model's maximum context length
model_name = "Qwen/Qwen2.5-Math-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_context_length = tokenizer.model_max_length
print(f"Maximum context length: {max_context_length}")

# Use smaller chunks to further reduce memory pressure
chunk_size = 3072  # Further reduced from 4096

# 3. Tokenize the entire corpus and split it into reasonably sized chunks
def prepare_corpus_for_training(corpus, tokenizer, chunk_size):
    tokens = tokenizer(corpus, truncation=False, return_tensors="np")["input_ids"][0]
    
    total_chunks = math.ceil(len(tokens) / chunk_size)
    print(f"Total tokens: {len(tokens)}, Creating {total_chunks} chunks of size {chunk_size}")
    
    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk = tokens[i:i + chunk_size].tolist()
        if len(chunk) < chunk_size:
            chunk = chunk + [tokenizer.pad_token_id] * (chunk_size - len(chunk))
        chunks.append({"input_ids": chunk})
    
    return Dataset.from_list(chunks)

chunked_dataset = prepare_corpus_for_training(corpus, tokenizer, chunk_size)

def apply_layer_pruning(model, pruning_ratio=0.3):
    """Apply structured pruning to even-numbered transformer layers"""
    for name, module in model.named_modules():
        if "transformer.h." in name and isinstance(module, torch.nn.Linear):
            layer_num = int(name.split(".")[2])
            if layer_num % 2 == 0:  # Only prune even-numbered layers
                prune.l1_unstructured(module, name='weight', amount=pruning_ratio)
    return model
    
# 4. Create a unified progress tracking callback
class EnhancedProgressCallback(TrainerCallback):
    def __init__(self):
        self.training_start = time.time()
        self.epoch_start = None
        self.progress_bar = None
        self.current_epoch = 0
        
    def on_train_begin(self, args, state, control, **kwargs):
        print(f"\n{'='*70}")
        print(f"TRAINING STARTED")
        print(f"{'='*70}")
        
        # Calculate total steps for all epochs
        self.total_steps = state.max_steps
        self.steps_per_epoch = len(trainer.train_dataset) // (args.per_device_train_batch_size * 
                                                            args.gradient_accumulation_steps * 
                                                            torch.cuda.device_count())  # Account for multi-GPU
    
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start = time.time()
        self.current_epoch = state.epoch + 1
        
        print(f"\n{'='*70}")
        print(f"Beginning Epoch {self.current_epoch}/{args.num_train_epochs}")
        print(f"{'='*70}")
        
        # Create a progress bar for this epoch
        self.progress_bar = tqdm(
            total=self.steps_per_epoch, 
            desc=f"Epoch {self.current_epoch}/{args.num_train_epochs}",
            position=0
        )
        self.last_logged_step = 0
        
    def on_epoch_end(self, args, state, control, **kwargs):
        # Close progress bar
        if self.progress_bar:
            self.progress_bar.close()
        
        # Calculate epoch time
        epoch_time = time.time() - self.epoch_start
        total_time = time.time() - self.training_start
        
        print(f"\n{'='*70}")
        print(f"Completed Epoch {self.current_epoch}/{args.num_train_epochs}")
        print(f"Epoch time: {timedelta(seconds=int(epoch_time))}")
        print(f"Total training time: {timedelta(seconds=int(total_time))}")
        
        # Estimate remaining time
        epochs_remaining = args.num_train_epochs - self.current_epoch
        est_remaining = epoch_time * epochs_remaining
        print(f"Estimated time remaining: {timedelta(seconds=int(est_remaining))}")
        print(f"{'='*70}\n")
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero and logs and self.progress_bar:
            # Calculate step progress
            current_step_in_epoch = state.global_step % self.steps_per_epoch
            if current_step_in_epoch == 0 and state.global_step > 0:
                current_step_in_epoch = self.steps_per_epoch
                
            # Update progress bar to current position
            self.progress_bar.n = current_step_in_epoch
            
            # Add metrics to progress bar
            postfix_dict = {}
            if "loss" in logs:
                postfix_dict["loss"] = f"{logs['loss']:.4f}"
            if "learning_rate" in logs:
                postfix_dict["lr"] = f"{logs['learning_rate']:.2e}"
            
            # Add GPU memory usage
            try:
                allocated = torch.cuda.memory_allocated() / (1024 ** 3)
                postfix_dict["GPU"] = f"{allocated:.1f}GB"
            except:
                pass
                
            self.progress_bar.set_postfix(**postfix_dict)
            self.progress_bar.update(0)  # Force refresh
            
            # Print step details with percentage completed
            current_overall = state.global_step
            progress_percent = current_overall / self.total_steps * 100
            if current_overall % 20 == 0:  # Print every 20 steps
                print(f"Step: {current_overall}/{self.total_steps} ({progress_percent:.1f}%) | "
                      f"Loss: {logs.get('loss', 0):.4f} | "
                      f"LR: {logs.get('learning_rate', 0):.2e}")
    
# Create data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 5. LoRA-optimized training arguments for continued pretraining
training_args = TrainingArguments(
    output_dir="./qwen_math_nbody_lora",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    # Standard learning rate for non-embedding layers
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    dataloader_drop_last=True,
    report_to=["tensorboard"],
    fp16=True,
    logging_first_step=True,
    gradient_checkpointing=True,
    logging_dir="./logs",
    warmup_ratio=0.1,
    logging_strategy="steps",
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # Use adamw_hf which better supports parameter groups for decoupled learning rates
    optim="adamw_8bit" 
)

# 6. Load model for examination to identify all module names
print("Loading model to identify layer structure...")
temp_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    use_cache=False,
    torch_dtype=torch.float16,
    device_map={"": 0}  # Load on first GPU only temporarily
)

# Identify embedding and other linear layers for targeting
embedding_layers = []
linear_layers = []
for name, module in temp_model.named_modules():
    if 'embed' in name.lower() or 'lm_head' in name.lower():
        embedding_layers.append(name)
    elif isinstance(module, torch.nn.Linear) and 'embed' not in name.lower() and 'lm_head' not in name.lower():
        linear_layers.append(name)

print(f"Found {len(embedding_layers)} embedding layers: {embedding_layers}")
print(f"Found {len(linear_layers)} linear layers")

# Clean up memory
del temp_model
torch.cuda.empty_cache()
gc.collect()

# 7. Comprehensive LoRA configuration for continued pretraining
# Combine both standard target_modules and add embedding layers
# List adjusted based on Qwen2.5 specific architecture

book_lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

paper_lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 8. Initialize model with optimized settings
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    use_cache=False,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.float16
)

model = apply_layer_pruning(model)
model = prepare_model_for_kbit_training(model)

# 9. Apply LoRA to the model
print("Applying LoRA adapters to model...")
model = get_peft_model(model, paper_lora_config, adapter_name="paper_adapter")
model.print_trainable_parameters()

# 10. Create custom optimizer with decoupled learning rates
def get_optimizer_grouped_parameters(model, embedding_lr=1e-5, non_embedding_lr=1e-4):
    """Create parameter groups with different learning rates for embeddings vs other layers."""
    no_decay = ["bias", "LayerNorm.weight"]
    embedding_names = ["wte", "lm_head"]
    
    optimizer_grouped_parameters = [
        # Embedding params with lower learning rate and no weight decay
        {
            "params": [p for n, p in model.named_parameters() 
                      if any(nd in n for nd in embedding_names) and p.requires_grad],
            "lr": embedding_lr,
            "weight_decay": 0.0,
        },
        # Non-embedding params with regular learning rate and weight decay
        {
            "params": [p for n, p in model.named_parameters() 
                      if not any(nd in n for nd in embedding_names) 
                      and not any(nd in n for nd in no_decay) and p.requires_grad],
            "lr": non_embedding_lr,
            "weight_decay": training_args.weight_decay,
        },
        # Non-embedding params with regular learning rate and no weight decay
        {
            "params": [p for n, p in model.named_parameters() 
                      if not any(nd in n for nd in embedding_names) 
                      and any(nd in n for nd in no_decay) and p.requires_grad],
            "lr": non_embedding_lr,
            "weight_decay": 0.0,
        },
    ]
    return optimizer_grouped_parameters

# 11. Create a custom trainer with decoupled learning rates
class CustomPEFTTrainer(Trainer):
    def create_optimizer(self):
        """Create optimizer with separate learning rates for embedding vs. non-embedding parameters"""
        if self.optimizer is None:
            # Create parameter groups with decoupled learning rates
            embedding_lr = self.args.learning_rate / 10  # Lower embedding LR (10x smaller)
            non_embedding_lr = self.args.learning_rate
            
            print(f"Using decoupled learning rates: embedding={embedding_lr}, other={non_embedding_lr}")
            
            optimizer_grouped_parameters = get_optimizer_grouped_parameters(
                self.model, 
                embedding_lr=embedding_lr,
                non_embedding_lr=non_embedding_lr
            )
            
            # Create optimizer with parameter groups
            self.optimizer = torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(0.9, 0.999),
                eps=1e-8,
            )
        
        return self.optimizer

# 12. Initialize custom trainer with all optimizations
progress_callback = EnhancedProgressCallback()
trainer = CustomPEFTTrainer(
    model=model,
    args=training_args,
    train_dataset=chunked_dataset,
    data_collator=data_collator,
)
trainer.add_callback(progress_callback)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Maximum context length: 131072


...

Total tokens: 1399402, Creating 456 chunks of size 3072
Loading model to identify layer structure...


config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Found 2 embedding layers: ['model.embed_tokens', 'lm_head']
Found 196 linear layers
Applying LoRA adapters to model...
trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [3]:

# Print training configuration
print(f"\n{'*'*70}")
print(f"ENHANCED LORA CONTINUED PRETRAINING CONFIGURATION:")
print(f"Model: {model_name}")
print(f"LoRA rank: {book_lora_config.r}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Using decoupled learning rates: embedding={training_args.learning_rate/10}, other={training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size} × "
      f"{training_args.gradient_accumulation_steps} steps × "
      f"{torch.cuda.device_count()} GPUs = "
      f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * torch.cuda.device_count()}")
print(f"Dataset size: {len(chunked_dataset)} chunks of {chunk_size} tokens each")
print(f"{'*'*70}\n")

# Start the training process
print("\n🚀 Starting research-optimized LoRA continued pretraining...\n")
try:
    trainer.train()
    print("\n✅ Training completed successfully!")
    # Save the final model
    print("Saving final model...")
    model.save_pretrained("./qwen_math_nbody_final_lora")
    tokenizer.save_pretrained("./qwen_math_nbody_final_lora")
    print("Model saved at ./qwen_math_nbody_final_lora")
except Exception as e:
    print(f"\n❌ Training interrupted: {e}")
    # Save checkpoint even if interrupted
    print("Saving emergency checkpoint...")
    model.save_pretrained("./qwen_math_nbody_checkpoint_lora")
    tokenizer.save_pretrained("./qwen_math_nbody_checkpoint_lora")
    print("Emergency checkpoint saved at ./qwen_math_nbody_checkpoint_lora")



**********************************************************************
ENHANCED LORA CONTINUED PRETRAINING CONFIGURATION:
Model: Qwen/Qwen2.5-Math-1.5B
LoRA rank: 8
Epochs: 3
Using decoupled learning rates: embedding=2e-05, other=0.0002
Batch size: 1 × 16 steps × 1 GPUs = 16
Dataset size: 456 chunks of 3072 tokens each
**********************************************************************


🚀 Starting research-optimized LoRA continued pretraining...

Using decoupled learning rates: embedding=2e-05, other=0.0002

TRAINING STARTED

Beginning Epoch 1/3


Epoch 1/3:   0%|          | 0/28 [00:00<?, ?it/s]

Step,Training Loss
1,14.063200
10,15.807900
20,14.594300
30,14.101600
40,15.343500
50,14.328000
60,12.904200
70,14.188300
80,14.000400


Step: 20/84 (23.8%) | Loss: 14.5943 | LR: 1.71e-05

Completed Epoch 1/3
Epoch time: 1:01:00
Total training time: 1:01:04
Estimated time remaining: 2:02:00


Beginning Epoch 2.0/3


Epoch 2.0/3:   0%|          | 0/28 [00:00<?, ?it/s]

Step: 40/84 (47.6%) | Loss: 15.3435 | LR: 1.17e-05

Completed Epoch 2.0/3
Epoch time: 1:00:58
Total training time: 2:02:03
Estimated time remaining: 1:00:58


Beginning Epoch 3.0/3


Epoch 3.0/3:   0%|          | 0/28 [00:00<?, ?it/s]

Step: 60/84 (71.4%) | Loss: 12.9042 | LR: 6.40e-06
Step: 80/84 (95.2%) | Loss: 14.0004 | LR: 1.07e-06

Completed Epoch 3.0/3
Epoch time: 0:55:38
Total training time: 2:57:41
Estimated time remaining: 0:00:00


✅ Training completed successfully!
Saving final model...
Model saved at ./qwen_math_nbody_final_lora


In [4]:
# !pip install transformers datasets ipywidgets peft bitsandbytes accelerate

In [5]:
# import os
# import json
# import shutil
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # 1. Load the base model
# base_model_id = "Qwen/Qwen2.5-Math-1.5B"
# model = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     torch_dtype="auto",
#     device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# # 2. Create a temporary directory to combine both adapter files
# temp_adapter_dir = "/kaggle/working/combined_adapter"
# os.makedirs(temp_adapter_dir, exist_ok=True)

# # 3. Copy the adapter weights file
# weights_source = "/kaggle/input/central_config_adapter/pytorch/default/1/adapter_model.safetensors"
# weights_dest = os.path.join(temp_adapter_dir, "adapter_model.safetensors")
# shutil.copy(weights_source, weights_dest)

# # 4. Copy or create the config file
# # Assuming you've uploaded the config JSON to a different location
# config_source = "/kaggle/input/adapter-config/adapter_config.json"  # Update this path
# config_dest = os.path.join(temp_adapter_dir, "adapter_config.json")
# shutil.copy(config_source, config_dest)

# # 5. Load the model with the combined adapter
# model = PeftModel.from_pretrained(model, temp_adapter_dir)

# # 6. Test the model
# prompt = "What are four body central configurations? How many four body central configurations exist?"
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=2048,  # Increased for more detailed explanation
#     temperature=0.6,     # Lower temperature for more precise outputs
#     # do_sample=True,
#     # top_p=0.95
# )
# response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(response)

In [6]:
# prompt = '''Explore the unique properties of spiderweb central configurations as defined in celestial mechanics, where masses lie at intersection points of concentric circles with lines meeting at equal angles at the center. Based on the available information:

# 1. Explain how the mass distribution in spiderweb configurations differs from other central configurations
# 2. Analyze Saari's findings about rotation curves and mass distribution approximations in these configurations
# 3. Discuss the potential applications of spiderweb central configurations for understanding galactic structures
# 4. Identify specific open questions about mass distribution in spiderweb configurations that merit further study

# Focus specifically on the spiderweb geometry and avoid generic discussions of n-body problems that don't relate to this specific configuration.'''
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=2048,  # Increased for more detailed explanation
#     temperature=0.6,     # Lower temperature for more precise outputs
#     # do_sample=True,
#     # top_p=0.95
# )
# response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(response)

In [7]:
# prompt = "Explain the concept of bifurcation and the stacking of central configurations in the planar $1+4$ body problem. Include relevant mathematical equations or expressions."
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=2048,  # Increased for more detailed explanation
#     temperature=0.6,     # Lower temperature for more precise outputs
#     # do_sample=True,
#     # top_p=0.95
# )
# response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(response)

In [8]:
# prompt = (
#     "You are an expert on central configurations in mathematical physics. "
#     "Please explain what co-circular central configurations are and show one derivation equation step-by-step."
# )

# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=2048,  # Increased for more detailed explanation
#     temperature=0.7,     # Lower temperature for more precise outputs
#     do_sample=True,
#     top_p=0.95,
#     repetition_penalty = 1.1
# )
# response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(response)

In [9]:
# # Load model directly
# from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("facebook/galactica-1.3b")
# model = AutoModelForCausalLM.from_pretrained("facebook/galactica-1.3b")

In [10]:
# # Move the model to GPU
# model = model.to("cuda")

# input_text = ("Explain the concept of bifurcation and the stacking of central configurations in the planar "
#               "$1+4$ body problem. Include relevant mathematical equations or expressions.")
# # Transfer input tokens to GPU
# input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

# # Generate output tokens
# outputs = model.generate(input_ids, max_length=1000)

# # Decode and print the generated text
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))